# Section 2 - Cache Frame Features

Run the frozen ResNet50 over every clip once and save the per-frame features.

```
Diving_01.pt   (16, 3, 224, 224)   9.63 MB
      |  frozen ResNet50, one time
      v
Diving_01      (16, 2048)          65.5 KB      147x smaller
```

The ResNet is frozen after Section 1, so it returns the same answer for the same clip
every epoch. Caching makes training ~40x faster (2 s/epoch instead of 80 s).

We cache at 2048 (before the LSTM) rather than the LSTM's 256-d output, so the LSTM
stays trainable.

## Setup

In [ ]:
import os, sys, subprocess
from getpass import getpass

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "embeddings"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print(subprocess.run(["git", "-C", ROOT, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
os.environ["KAGGLE_USERNAME"] = "YOUR_KAGGLE_USER"
os.environ["KAGGLE_KEY"]      = "YOUR_KAGGLE_KEY"

from src.bootstrap import setup
setup(drive=True, dataset=True)

In [ ]:
%run /content/ulal/src/imports.py

In [ ]:
TASK_CLASSES = [cfg.SELECTED_CLASSES, cfg.TASK_1, cfg.TASK_2, cfg.TASK_4]
TASK_ROOTS   = [cfg.BASE_ROOT, cfg.TASK1_ROOT, cfg.TASK2_ROOT, cfg.TASK4_ROOT]
TASK_NAMES   = ["Base", "Task1", "Task2", "Task4"]
CL_TASKS     = [1, 2]

global_classes, global_c2i, NUM_AFTER = build_label_space(TASK_CLASSES)
describe_label_space(TASK_CLASSES, TASK_NAMES, NUM_AFTER, task_3=cfg.TASK_3)

## Preprocess

In [ ]:
FORCE_REEXTRACT = False

NEED = FORCE_REEXTRACT or not os.path.exists(cfg.FEAT_CACHE)
print("Extracting" if NEED else "Cache exists, skipping")

if NEED:
    for root, classes, name in zip(TASK_ROOTS, TASK_CLASSES, TASK_NAMES):
        print(f"\n{name}: {len(classes)} classes -> {root}")
        cfg.OUTPUT_ROOT = root
        preprocess_dataset(splits=["train", "val", "test"], target_classes=classes)

## Load and freeze the teacher

In [ ]:
teacher = None
if NEED:
    teacher = ResNet50LSTMTeacher(
        hidden_size = cfg.TEACHER_HIDDEN,
        num_classes = len(cfg.SELECTED_CLASSES),
        dropout_p   = cfg.DROPOUT_P,
    ).to(device)

    mapped = remap_teacher_checkpoint(torch.load(cfg.RESNET50_PATH, map_location=device))
    missing, _ = teacher.load_state_dict(mapped, strict=False)
    real_missing = [k for k in missing if "num_batches_tracked" not in k]
    if real_missing:
        raise RuntimeError(f"Checkpoint missing keys: {real_missing[:5]}")

    for p in teacher.parameters():
        p.requires_grad_(False)
    teacher.eval()
    print("Teacher frozen")

## Extract

Each split is cached separately: train fits, val selects, test reports.

In [ ]:
if NEED:
    FEATS = {}
    for t, (root, classes, name) in enumerate(zip(TASK_ROOTS, TASK_CLASSES, TASK_NAMES)):
        for split in ["train", "val", "test"]:
            ds, loader = build_clip_loader(root, classes, global_c2i, split,
                                           batch_size=cfg.BATCH_SIZE,
                                           num_workers=cfg.NUM_WORKERS,
                                           pin_memory=(device.type == "cuda"))
            if len(ds) == 0:
                print(f"  {name}/{split} empty")
                continue
            f, y = extract_frame_features(teacher.backbone, loader, device,
                                          backbone_dim=cfg.BACKBONE_DIM,
                                          dtype=getattr(torch, cfg.STORE_DTYPE))
            FEATS[f"t{t}_{split}"] = (f, y)
            print(f"  {name:<6} {split:<5}: {tuple(f.shape)}")

    save_feature_cache(cfg.FEAT_CACHE, FEATS, teacher.lstm.state_dict(),
                       cfg.BACKBONE_DIM, cfg.CLIP_LEN, global_classes, TASK_NAMES)
    del teacher
    teacher = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Verify

In [ ]:
CACHE, LSTM_STATE, META = load_feature_cache(cfg.FEAT_CACHE)
for k in sorted(CACHE):
    f, y = CACHE[k]
    print(f"  {k:<12}: {str(tuple(f.shape)):<22} {len(set(y.tolist())):>2} classes")

verify_cache(CACHE, TASK_CLASSES, global_c2i, cfg.BACKBONE_DIM, cfg.CLIP_LEN)

## Are the features good enough?

The ResNet was fine-tuned on 10 sports classes then frozen. This checks whether it still
carries the new classes. We mean-pool each clip to (2048,) and ask a simple classifier to
separate the classes.

If the joint probe is high, the features are fine and any later accuracy drop is
forgetting rather than a bad representation. The joint linear score is also the
reference ceiling used by the Section 4 notebooks.

In [ ]:
report = probe_report(CACHE, TASK_NAMES, k=1)

## Cluster geometry

Classes that sit close together will compete during continual learning.

In [ ]:
task_ids = [t for t in range(len(TASK_NAMES)) if f"t{t}_train" in CACHE]

print(f"{'Group':<10}{'silhouette':>12}")
for t in task_ids:
    s = separability(CACHE, [t])
    if s is not None:
        print(f"{TASK_NAMES[t]:<10}{s:>12.3f}")
s_joint = separability(CACHE, task_ids)
print(f"{'JOINT':<10}{s_joint:>12.3f}")

cents        = class_centroids(CACHE, task_ids, "train")
labels, dist = centroid_distances(cents)

print("\nClosest class pairs:")
for d, a, b in closest_pairs(cents, global_classes, top=10):
    ta = next(i for i, g in enumerate(TASK_CLASSES) if a in g)
    tb = next(i for i, g in enumerate(TASK_CLASSES) if b in g)
    flag = "  <- different tasks" if ta != tb else ""
    print(f"  {d:7.2f}  {a:<22} {b:<22} [{TASK_NAMES[ta]}/{TASK_NAMES[tb]}]{flag}")

## Plots

In [ ]:
task_of = {global_c2i[c]: t for t, g in enumerate(TASK_CLASSES) for c in g}
xy, lab = project_2d(CACHE, task_ids, split="train", method="tsne", seed=cfg.SEED)

fig = plt.figure(figsize=(16, 11))
gs  = fig.add_gridspec(2, 2, hspace=0.25, wspace=0.2)

plot_probe_bars(report, ax=fig.add_subplot(gs[0, 0]))
plot_latent_space(xy, lab, global_classes, task_of=task_of,
                  title="Latent space by task", ax=fig.add_subplot(gs[0, 1]))
plot_latent_space(xy, lab, global_classes,
                  title="Latent space by class", ax=fig.add_subplot(gs[1, 0]))
plot_centroid_heatmap(labels, dist, global_classes, ax=fig.add_subplot(gs[1, 1]))

plt.savefig(f"{cfg.RESULTS_DIR}/stage2_embedding_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()

## Save

In [ ]:
diag = {
    "stage": "2_embedding_quality",
    "probe": report,
    "silhouette": {TASK_NAMES[t]: separability(CACHE, [t]) for t in task_ids},
    "silhouette_joint": s_joint,
    "closest_pairs": [{"dist": float(d), "a": a, "b": b}
                      for d, a, b in closest_pairs(cents, global_classes, top=10)],
    "n_clips": int(sum(f.shape[0] for f, _ in CACHE.values())),
}
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
with open(f"{cfg.RESULTS_DIR}/stage2_embedding_diagnostics.json", "w") as f:
    json.dump(diag, f, indent=2)

print(f"JOINT linear probe: {report['JOINT']['linear']:.2%}  (ceiling for Section 4)")

---

Next: `4_continual_learning/`. Run 4.1 first (it trains and saves the shared base head),
then 4.2 to 4.6 in any order.